# 02 - Baseline de ML clasico

Pipeline: extraemos features con un backbone preentrenado (ResNet18 frozen)
y entrenamos clasificadores clasicos de scikit-learn.

Etapas:
1. Setup, config y data loaders.
2. Extraccion de features con ResNet18 (sin entrenamiento).
3. Entrenamiento y comparacion de modelos sklearn.
4. Evaluacion en test (accuracy, F1 macro, classification report).
5. Persistencia del mejor modelo y logging opcional a W&B.

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from dotenv import load_dotenv
load_dotenv(PROJECT_ROOT / ".env")

from src.utils.config import load_yaml_config
from src.utils.reproducibility import set_global_seed

import joblib
import numpy as np
from torch import nn
from torchvision import models

from src.utils.wandb_utils import finish_wandb_run, init_wandb_run

CONFIG_PATH = PROJECT_ROOT / "configs" / "baseline_ml.yaml"
config = load_yaml_config(CONFIG_PATH)
set_global_seed(config["seed"])
import torch
device = torch.device("cuda" if torch.cuda.is_available() else config["device"])
print("Device:", device)
config

## Data loaders

In [ ]:
import torch
from torch.utils.data import DataLoader
from torchvision import datasets

from src.data.dataset import build_image_transforms, load_imagefolder_datasets

data_root = PROJECT_ROOT / config["data"]["root_dir"]
image_size = config["data"]["image_size"]
batch_size = config["data"]["batch_size"]
num_workers = config["data"].get("num_workers", 0)

train_dataset, val_dataset = load_imagefolder_datasets(
    root_dir=data_root,
    train_subdir=config["data"].get("train_subdir", "train"),
    val_subdir=config["data"].get("val_subdir", "val"),
    image_size=image_size,
)

test_dir = data_root / config["data"].get("test_subdir", "test")
test_transform = build_image_transforms(image_size=image_size)
test_dataset = datasets.ImageFolder(root=test_dir, transform=test_transform)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=num_workers)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers)

class_names = train_dataset.classes
num_classes = len(class_names)
print({"train": len(train_dataset), "val": len(val_dataset), "test": len(test_dataset), "num_classes": num_classes})


## Extraccion de features con ResNet18 (frozen)

In [ ]:
backbone = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
backbone.fc = nn.Identity()
backbone.eval().to(device)
for p in backbone.parameters():
    p.requires_grad = False

@torch.no_grad()
def extract_features(loader):
    feats, labels = [], []
    for images, targets in loader:
        images = images.to(device, non_blocking=True)
        feats.append(backbone(images).cpu().numpy())
        labels.append(targets.numpy())
    return np.concatenate(feats), np.concatenate(labels)

X_train, y_train = extract_features(train_loader)
X_val, y_val = extract_features(val_loader)
X_test, y_test = extract_features(test_loader)
print("Feature shape:", X_train.shape)

## Entrenamiento de modelos sklearn

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, f1_score
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC

scaler = StandardScaler().fit(X_train)
X_train_s = scaler.transform(X_train)
X_val_s = scaler.transform(X_val)
X_test_s = scaler.transform(X_test)

candidates = {
    "logreg": LogisticRegression(max_iter=2000, n_jobs=-1, class_weight="balanced"),
    "linear_svc": LinearSVC(class_weight="balanced"),
    "random_forest": RandomForestClassifier(
        n_estimators=200, n_jobs=-1, class_weight="balanced", random_state=config["seed"]
    ),
}

results = {}
for name, model in candidates.items():
    model.fit(X_train_s, y_train)
    val_preds = model.predict(X_val_s)
    results[name] = {
        "model": model,
        "val_accuracy": accuracy_score(y_val, val_preds),
        "val_f1_macro": f1_score(y_val, val_preds, average="macro"),
    }
    print(f"{name}: acc={results[name]['val_accuracy']:.4f} f1_macro={results[name]['val_f1_macro']:.4f}")

## Evaluacion en test del mejor modelo

In [ ]:
best_name = max(results, key=lambda k: results[k]["val_f1_macro"])
best = results[best_name]["model"]
print("Best model:", best_name)

test_preds = best.predict(X_test_s)
test_acc = accuracy_score(y_test, test_preds)
test_f1 = f1_score(y_test, test_preds, average="macro")
print("Test accuracy:", test_acc)
print("Test F1 macro:", test_f1)
print(classification_report(y_test, test_preds, target_names=class_names))

## Guardado del modelo y logging W&B (opcional)

In [ ]:
output_dir = PROJECT_ROOT / config["output"]["artifacts_dir"] / "baseline_ml"
output_dir.mkdir(parents=True, exist_ok=True)
model_path = output_dir / config["output"]["model_name"]
joblib.dump({"model": best, "scaler": scaler, "class_names": class_names, "best_name": best_name}, model_path)
print("Saved best model to", model_path)

run = init_wandb_run(
    config=config,
    enabled=config["tracking"].get("use_wandb", False),
    project=config["tracking"]["project"],
    run_name=config["tracking"].get("run_name", config["experiment_name"]),
    tags=config["tracking"].get("tags"),
)
if run is not None:
    run.log({
        "best_model": best_name,
        "val_accuracy": results[best_name]["val_accuracy"],
        "val_f1_macro": results[best_name]["val_f1_macro"],
        "test_accuracy": test_acc,
        "test_f1_macro": test_f1,
    })
    finish_wandb_run(run)